In [1]:
# ============================================================
# Project: EuroTrans Analytics
# Notebook: 03_Gold_Delivery_Analytics
# Layer: Gold
#
# Description:
# Build the Gold Fact Shipment table for the semantic model.
# ============================================================

# ------------------------------------------------------------
# Import Libraries
# ------------------------------------------------------------

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, round

# ------------------------------------------------------------
# Create Spark Session
# ------------------------------------------------------------

spark = SparkSession.builder.getOrCreate()

print("=" * 60)
print("Loading Silver tables...")
print("=" * 60)

# ------------------------------------------------------------
# Load Silver Tables
# ------------------------------------------------------------

shipment = spark.table("silver_shipment")
customer = spark.table("silver_customer")
product = spark.table("silver_product")
carrier = spark.table("silver_carrier")
route = spark.table("silver_route")

# ------------------------------------------------------------
# Create Gold Dataset
# ------------------------------------------------------------

gold = (
    shipment
        .join(customer, "CustomerID", "left")
        .join(product, "ProductID", "left")
        .join(carrier, "CarrierID", "left")
        .join(route, "RouteID", "left")
)

# ------------------------------------------------------------
# Business Metrics
# ------------------------------------------------------------

gold = gold.withColumn(
    "ProfitMarginPct",
    when(
        col("RevenueEUR") > 0,
        round(
            col("ProfitEUR") / col("RevenueEUR"),
            4
        )
    ).otherwise(0)
)

gold = gold.withColumn(
    "RevenuePerKg",
    when(
        col("WeightKg") > 0,
        round(
            col("RevenueEUR") / col("WeightKg"),
            2
        )
    ).otherwise(0)
)

gold = gold.withColumn(
    "CostPerKg",
    when(
        col("WeightKg") > 0,
        round(
            col("TotalTransportCostEUR") / col("WeightKg"),
            2
        )
    ).otherwise(0)
)

gold = gold.withColumn(
    "IsDelayed",
    when(
        col("DelayHours") > 0,
        "Yes"
    ).otherwise("No")
)

# ------------------------------------------------------------
# Select Business Columns
# ------------------------------------------------------------

gold_fact = gold.select(

    # Keys
    "ShipmentID",
    "ShipmentDate",
    "CustomerID",
    "ProductID",
    "CarrierID",
    "RouteID",

    # Measures
    "WeightKg",
    "RevenueEUR",

    "FuelCostEUR",
    "TollCostEUR",
    "DriverCostEUR",
    "MaintenanceCostEUR",
    "HandlingCostEUR",

    "TotalTransportCostEUR",
    "ProfitEUR",
    "ProfitMarginPct",

    # KPIs
    "RevenuePerKg",
    "CostPerKg",

    "DeliveryStatus",
    "DelayHours",
    "IsDelayed"
)

# ------------------------------------------------------------
# Save Gold Fact Table
# ------------------------------------------------------------

(
    gold_fact.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_fact_shipments")
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 60)
print("Gold Fact Shipments created successfully.")
print("=" * 60)

print(f"Rows: {gold_fact.count()}")
print(f"Columns: {len(gold_fact.columns)}")

display(gold_fact.limit(10))

StatementMeta(, 1a4329d6-5957-493d-b2b6-09196e4d9baf, 3, Finished, Available, Finished, False)

Loading Silver tables...
Gold Fact Shipments created successfully.
Rows: 50000
Columns: 21


SynapseWidget(Synapse.DataFrame, 36b9988d-ea62-4bb6-bdf0-c7a153d6995a)